# Módulo 2 · Visualizar EEG con MNE

**Objetivo:** ver la señal como la vería un neurólogo, mirar su espectro de frecuencias y
situar los electrodos en la cabeza. Al final **verás la descarga punta-onda a 3 Hz**
característica de la crisis de ausencia.

### Interactivo vs. estático
- En Jupyter, por defecto las figuras son **estáticas** (imagen).
- Para navegar por la señal con scroll, teclado y zoom, ejecuta en una celda
  `%matplotlib qt` (o instala `mne-qt-browser`, que MNE usa automáticamente).
  Atajos útiles en `raw.plot()`: ←/→ avanzar en el tiempo, ↑/↓ cambiar canales,
  `+`/`-` escala, `a` modo anotación, `?` ayuda.
- En este notebook forzamos el *backend* matplotlib para que las figuras queden guardadas.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, set_1020, TUSZ_ROOT

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

raw = read_tusz_raw("aaaaabqv_s001_t000", annotate=False)  # lo que construiste en el módulo 1
raw.pick("eeg")
raw

## 2.1 `raw.plot()`: la vista clásica

Parámetros clave:
- `start`, `duration`: ventana temporal (s)
- `n_channels`: canales visibles a la vez
- `scalings`: amplitud que ocupa la separación entre canales. `dict(eeg=100e-6)` = 100 µV.
  También `scalings="auto"`.
- `order`: orden de los canales

In [ ]:
fig = raw.plot(start=170, duration=25, n_channels=22, scalings=dict(eeg=150e-6),
               title="aaaaabqv · alrededor de la 1ª crisis de ausencia", show_scrollbars=False)

Alrededor de **t = 180 s** aparece en *todos* los canales a la vez una actividad rítmica
de gran amplitud: es la descarga **punta-onda generalizada a ~3 Hz** de la ausencia. Que
aparezca en todos los canales simultáneamente es lo que la hace **generalizada**.

Con filtro de visualización (no modifica los datos) se ve más limpia:

In [ ]:
fig = raw.plot(start=176, duration=15, n_channels=22, scalings=dict(eeg=150e-6),
               highpass=1.0, lowpass=30.0, show_scrollbars=False,
               title="Mismo tramo, filtrado de visualización 1-30 Hz")

## 2.2 Tu propia figura con matplotlib

`raw.plot()` es genial para explorar, pero para la memoria del TFM querrás figuras a medida.
Patrón: `get_data()` + matplotlib.

In [ ]:
chs = ["FP1", "F3", "C3", "P3", "O1", "FP2", "F4", "C4", "P4", "O2"]
x, t = raw.copy().filter(1, 30).get_data(picks=chs, tmin=176, tmax=192, return_times=True, units="uV")

fig, ax = plt.subplots(figsize=(12, 6))
offset = 250  # µV entre trazas
for i, ch in enumerate(chs):
    ax.plot(t, x[i] - i * offset, lw=0.7, color="k")
    ax.text(t[0] - 0.3, -i * offset, ch, ha="right", va="center", fontsize=9)
ax.axvspan(179.884, 187.016, color="tab:red", alpha=0.12, label="crisis anotada (absz)")
ax.set_yticks([]); ax.set_xlabel("Tiempo (s)"); ax.legend(loc="upper right")
ax.set_title("Crisis de ausencia: punta-onda generalizada")
for s in ("left", "top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 2.3 Espectro de potencia (PSD)

`raw.compute_psd()` devuelve un objeto **`Spectrum`** (método Welch por defecto).
Después puedes dibujarlo (`.plot()`) o sacar los números (`.get_data()`).

In [ ]:
spec = raw.compute_psd(fmax=70)
fig = spec.plot(average=False, amplitude=False)

Fíjate en el **pico estrecho a 60 Hz**: es la red eléctrica. Temple University está en EE. UU.,
donde la red va a **60 Hz** (no 50 Hz como en España). Lo quitaremos en el módulo 5.

### Comparar el espectro dentro y fuera de la crisis
Es el gráfico más revelador para una ausencia:

In [ ]:
spec_crisis = raw.copy().crop(180, 187).compute_psd(fmin=0.5, fmax=30, n_fft=512)
spec_basal  = raw.copy().crop(100, 107).compute_psd(fmin=0.5, fmax=30, n_fft=512)

p_c, f = spec_crisis.get_data(return_freqs=True)
p_b, _ = spec_basal.get_data(return_freqs=True)

plt.figure(figsize=(8, 4))
plt.semilogy(f, p_b.mean(0) * 1e12, label="basal (100-107 s)")
plt.semilogy(f, p_c.mean(0) * 1e12, label="crisis (180-187 s)", color="tab:red")
plt.axvline(3, ls="--", c="gray"); plt.text(3.2, plt.ylim()[1] * 0.3, "3 Hz", color="gray")
plt.xlabel("Frecuencia (Hz)"); plt.ylabel("PSD (µV²/Hz)"); plt.legend()
plt.title("Media de canales EEG"); plt.tight_layout(); plt.show()

print("Pico durante la crisis:", f[np.argmax(p_c.mean(0))], "Hz")

## 2.4 Posiciones de los electrodos (montaje 10-20)

El EDF no trae coordenadas. Se las damos con un **montaje estándar 10-20**
(`set_1020` de `tusz_utils` usa `colin27_1020` o `standard_1020` según la versión de MNE).
TUSZ usa la nomenclatura antigua (T3/T4/T5/T6 en lugar de T7/T8/P7/P8); el montaje estándar
la reconoce. Los canales sin posición estándar (T1, T2) se descartan.

In [ ]:
set_1020(raw)
fig = raw.plot_sensors(show_names=True)

Con posiciones ya podemos hacer **mapas topográficos** de potencia por banda:

In [ ]:
bands = {"Delta (1-4 Hz)": (1, 4), "Theta (4-8 Hz)": (4, 8), "Alfa (8-13 Hz)": (8, 13)}
spec_cr = raw.copy().crop(180, 187).compute_psd(fmin=1, fmax=30, n_fft=512)
fig = spec_cr.plot_topomap(bands=bands, show=False)
fig.suptitle("Durante la crisis de ausencia"); plt.show()

## 2.5 Otras vistas útiles

- `raw.plot_psd_topo()` / `spec.plot_topo()` — un espectro por electrodo, colocado en la cabeza.
- `mne.viz.plot_raw_psd` — equivalente funcional.
- `raw.plot(..., group_by="position")` — ordena canales por región (requiere montaje).

In [ ]:
fig = raw.copy().crop(180, 187).compute_psd(fmin=1, fmax=30, n_fft=512).plot_topo(color="k", fig_facecolor="w", axis_facecolor="w")

---
## 🧪 Ejercicios

**E2.1** Dibuja con `raw.plot()` las **otras dos crisis** de este fichero
(empiezan en ~242.7 s y ~286.0 s). ¿Se parecen a la primera?

**E2.2** Repite la comparación de PSD crisis vs. basal para la 2ª crisis. ¿El pico sigue en ~3 Hz?

**E2.3** Haz un mapa topográfico **solo de la banda 2.5-3.5 Hz** durante la crisis.
¿Dónde es máxima la potencia? (Las ausencias suelen ser de predominio **frontal**.)

### Solución E2.1

In [ ]:
for t0 in (240, 284):
    raw.plot(start=t0, duration=12, n_channels=22, scalings=dict(eeg=150e-6),
             highpass=1, lowpass=30, show_scrollbars=False, title=f"Crisis en t≈{t0 + 2} s")

### Solución E2.2

In [ ]:
pc, f = raw.copy().crop(242.7, 249.9).compute_psd(fmin=0.5, fmax=30, n_fft=512).get_data(return_freqs=True)
print("Pico 2ª crisis:", f[np.argmax(pc.mean(0))], "Hz")

### Solución E2.3

In [ ]:
fig = raw.copy().crop(180, 187).compute_psd(fmin=1, fmax=30, n_fft=512) \
         .plot_topomap(bands={"2.5-3.5 Hz": (2.5, 3.5)})